In [1]:
import pandas as pd
import os
import polars as pl

In [88]:
from collections import defaultdict

In [32]:
import cobra

In [33]:
from cobra.io import read_sbml_model

In [2]:
proj_dir = "/home/users/gsantamaria/projects/metabognn_dev/"
proj_dir_scratch = "/scratch/users/gsantamaria/test_large_files/metabognn_dev/"

In [3]:
attene_f = os.path.join(
    proj_dir,
    "data/validation/attene_ramos/AID_720637_datatable.csv"
)
attene_drugs_f = os.path.join(
    proj_dir_scratch,
    "results/mc_phenotype_assoc/HEPG2/attene_mmp_down_drugs.tsv"
)

In [4]:
mc_dir = os.path.join(
    proj_dir_scratch, "results/metChange/HEPG2/rank_concat"
)

mc_f = os.path.join(mc_dir, "expr_ensembl_rank_metChange_rpmi_medium_concat.csv")

In [5]:
expr_dir = os.path.join(
    proj_dir_scratch, "data/perturbation/lincs/cell_matrices/HEPG2"
)
expr_f = os.path.join(expr_dir, "expr_ensembl.csv")
pnlt_f = os.path.join(expr_dir, "expr_ensembl_rank.csv")
mdat_f = os.path.join(expr_dir, "mdat.csv")

In [34]:
gem_f = os.path.join(proj_dir, "data/GEMs/Human-GEM/model/Human-GEM.xml")

In [36]:
gem = read_sbml_model(gem_f)

In [43]:
gem.reactions[24]

'Artificial reactions'

In [117]:
# Get unique subsystems to see what name does TCA cycle hold
uniq_subsysts = []
for rxn in gem.reactions:
    rxn_subsysts = rxn.subsystem
    if isinstance(rxn_subsysts, str):
        if rxn_subsysts not in uniq_subsysts:
            uniq_subsysts.append(rxn_subsysts)
    else:
        for subs in rxn_subsysts:
            if subs not in uniq_subsysts:
                uniq_subsysts.append(subs)

In [113]:
def get_metrxndict(gem, whatmets="products", subsystem="all"):
    metrxndict = defaultdict(list)
    for r in gem.reactions:
        if subsystem == "all":
            keep_rxn = True
        else:
            keep_rxn = r.subsystem == subsystem
        if keep_rxn:
            is_rev = (r.lower_bound < 0 and r.upper_bound > 0) or (r.lower_bound > 0 and r.upper_bound < 0)
            if is_rev:
                mets = r.metabolites
            else:
                mets = getattr(r, whatmets)
            for m in mets:
                if r.id not in metrxndict[p.id]:
                    metrxndict[m.id].append(r.id)
    metrxndict = dict(metrxndict)
    return metrxndict

In [116]:
prod2rxn_tca = get_metrxndict(gem, "products", subsystem='Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism')
reac2rxn_tca = get_metrxndict(gem, "reactants", subsystem='Tricarboxylic acid cycle and glyoxylate/dicarboxylate metabolism')

In [127]:
list(gem.reactions[100].genes)[0].id

'ENSG00000147383'

In [128]:
def parse_metrnxndict(metrxndict, gem=gem):
    ids = []
    names = []
    formulas = []
    compartments = []
    reactions = []
    genes = []
    for met in list(metrxndict.keys()):
        m = gem.metabolites.get_by_id(met)
        ids.append(m.id)
        names.append(m.name)
        formulas.append(m.formula)
        compartments.append(m.formula)
        rxns = metrxndict[met]
        reactions.append('|'.join(rxns))
        g = []
        for r in rxns:
            for gene in list(gem.reactions.get_by_id(r).genes):
                g.append(gene.id)
        g = '|'.join(g)     
        genes.append(g)
    df = pd.DataFrame({
        "id": ids,
        "name": names,
        "formula": formulas,
        "compartment": compartments,
        "reactions": reactions,
        "genes": genes
    })
    return df

In [129]:
prod2rxn_tca_df = parse_metrnxndict(prod2rxn_tca, gem=gem)
reac2rxn_tca_df = parse_metrnxndict(reac2rxn_tca, gem=gem)

In [130]:
prod2rxn_tca_df

,id,name,formula,compartment,reactions,genes
0,MAM02633x,OAA,C4H2O5,C4H2O5,MAR08981,
1,MAM02183c,isocitrate,C6H5O7,C6H5O7,MAR00710|MAR04454,ENSG00000138413|ENSG00000122729
2,MAM02554c,NADP+,C21H25N7O17P3,C21H25N7O17P3,MAR00710|MAR04465,ENSG00000138413|ENSG00000137106
3,MAM01306c,AKG,C5H4O5,C5H4O5,MAR00710,ENSG00000138413
4,MAM01596c,CO2,CO2,CO2,MAR00710|MAR08774,ENSG00000138413
...,...,...,...,...,...,...
58,MAM02661c,oxalate,C2O4,C2O4,MAR08780,ENSG00000166796|ENSG00000171989|ENSG0000016680...
59,MAM00674c,2-phospho-D-glycerate,C3H4O7P,C3H4O7P,MAR08781,
60,MAM02439x,malate,C4H4O5,C4H4O5,MAR08981,
61,MAM02552x,NAD+,C21H26N7O14P2,C21H26N7O14P2,MAR08981,


In [6]:
attene_drugs = pd.read_csv(attene_drugs_f, sep = '\t')

In [7]:
(attene_drugs["labels_seen"] == "MMP_down").sum()

np.int64(104)

In [8]:
attene_drugs_down = attene_drugs[attene_drugs["group"] == "MMP_down"]

In [9]:
mdat = pd.read_csv(mdat_f)

In [10]:
attene_drugs_down

,drug,group,n_lincs_perts,n_records,labels_seen,potency_uM,efficacy_pct
0,AM-580,MMP_down,1,1,MMP_down,58.8300,-121.30
2,BRD-A29731977,MMP_down,1,2,MMP_down;cytotox,11.4400,-62.20
5,BRD-A66861218,MMP_down,1,1,MMP_down,28.7300,-95.85
8,BRD-K01815685,MMP_down,12,1,MMP_down,43.6600,-102.80
9,BRD-K09784055,MMP_down,6,2,MMP_down;incon,24.3400,-74.43
...,...,...,...,...,...,...,...
587,triciribine,MMP_down,7,1,MMP_down,1.2060,-62.57
588,triclosan,MMP_down,7,2,MMP_down,1.3770,-102.90
592,troglitazone,MMP_down,9,2,MMP_down,10.5900,-121.70
605,vinblastine,MMP_down,9,2,MMP_down,0.8873,-42.03


In [12]:
mdat["pertname"]

0          vorinostat
1           lapatinib
2        mitoxantrone
3         fulvestrant
4             HMN-214
             ...     
51450            DMSO
51451            DMSO
51452            DMSO
51453            DMSO
51454            DMSO
Name: pertname, Length: 51455, dtype: object

In [14]:
keep_perts = [x in attene_drugs_down["drug"].to_list() or x == "DMSO" for x in mdat["pertname"].to_list()]

In [28]:
keep_ids = mdat.loc[keep_perts, "id"].to_list()

In [17]:
mdat.loc[keep_perts, "id"].to_list()

3            ABY001_HEPG2_XH_X1_B15:N11
21           ABY001_HEPG2_XH_X1_B15:M03
25           ABY001_HEPG2_XH_X1_B15:D19
26           ABY001_HEPG2_XH_X1_B15:L11
29           ABY001_HEPG2_XH_X1_B15:D15
                      ...              
51450    POL001_HEPG2_24H_X1.L2_B23:F08
51451    POL001_HEPG2_24H_X1.L2_B23:A06
51452    POL001_HEPG2_24H_X3.L2_B23:J18
51453    POL001_HEPG2_24H_X3.L2_B23:B04
51454    POL001_HEPG2_24H_X3.L2_B23:J17
Name: id, Length: 5482, dtype: object

In [24]:
cols_mc = pl.scan_csv(mc_f).collect_schema().names()
mc = (
    pl.scan_csv(mc_f)
    .select([cols_expr[0]] + keep_ids)
    .collect()
)

In [29]:
cols_expr = pl.scan_csv(expr_f).collect_schema().names()

In [30]:
expr = (
    pl.scan_csv(expr_f)
    .select([cols_expr[0]] + keep_ids)
    .collect()
)

In [31]:
expr

ensembl_gene_id,ABY001_HEPG2_XH_X1_B15:N11,ABY001_HEPG2_XH_X1_B15:M03,ABY001_HEPG2_XH_X1_B15:D19,ABY001_HEPG2_XH_X1_B15:L11,ABY001_HEPG2_XH_X1_B15:D15,ABY001_HEPG2_XH_X1_B15:E15,ABY001_HEPG2_XH_X1_B15:N07,ABY001_HEPG2_XH_X1_B15:E23,ABY001_HEPG2_XH_X1_B15:L07,ABY001_HEPG2_XH_X1_B15:E19,ABY001_HEPG2_XH_X1_B15:L03,ABY001_HEPG2_XH_X1_B15:N03,ABY001_HEPG2_XH_X1_B15:M11,ABY001_HEPG2_XH_X1_B15:M07,ABY001_HEPG2_XH_X1_B15:F15,ABY001_HEPG2_XH_X1_B15:F23,ABY001_HEPG2_XH_X1_B15:D23,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:F03,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:D09,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:G11,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:I14,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:K09,CPC004_HEPG2_6H_X2_F1B5_DUO52HI53LO:E14,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:I14,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:K11,CPC004_HEPG2_6H_X2_F1B5_DUO52HI53LO:D12,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:D12,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:G09,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:L04,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:I07,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:J13,CPC004_HEPG2_6H_X2_F1B5_DUO52HI53LO:E07,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:L07,CPC004_HEPG2_6H_X3.L2_F2B5_DUO52HI53LO:F01,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:H08,CPC004_HEPG2_6H_X1_B5_DUO52HI53LO:E23,…,POL001_HEPG2_24H_X2.L2_B23:B06,POL001_HEPG2_24H_X1.L2_B23:J16,POL001_HEPG2_24H_X2.L2_B23:A04,POL001_HEPG2_24H_X2.L2_B23:J17,POL001_HEPG2_24H_X2.L2_B23:B04,POL001_HEPG2_24H_X3.L2_B23:J16,POL001_HEPG2_24H_X2.L2_B23:N24,POL001_HEPG2_24H_X2.L2_B23:F07,POL001_HEPG2_24H_X3.L2_B23:B03,POL001_HEPG2_24H_X2.L2_B23:F10,POL001_HEPG2_24H_X3.L2_B23:F08,POL001_HEPG2_24H_X2.L2_B23:F08,POL001_HEPG2_24H_X2.L2_B23:F09,POL001_HEPG2_24H_X3.L2_B23:J14,POL001_HEPG2_24H_X3.L2_B23:F07,POL001_HEPG2_24H_X3.L2_B23:F12,POL001_HEPG2_24H_X3.L2_B23:A03,POL001_HEPG2_24H_X2.L2_B23:J13,POL001_HEPG2_24H_X3.L2_B23:N24,POL001_HEPG2_24H_X1.L2_B23:J13,POL001_HEPG2_24H_X3.L2_B23:N21,POL001_HEPG2_24H_X3.L2_B23:A06,POL001_HEPG2_24H_X2.L2_B23:N23,POL001_HEPG2_24H_X1.L2_B23:B03,POL001_HEPG2_24H_X2.L2_B23:J18,POL001_HEPG2_24H_X2.L2_B23:J14,POL001_HEPG2_24H_X1.L2_B23:B06,POL001_HEPG2_24H_X2.L2_B23:J15,POL001_HEPG2_24H_X1.L2_B23:F12,POL001_HEPG2_24H_X1.L2_B23:B04,POL001_HEPG2_24H_X1.L2_B23:N21,POL001_HEPG2_24H_X3.L2_B23:F10,POL001_HEPG2_24H_X1.L2_B23:F08,POL001_HEPG2_24H_X1.L2_B23:A06,POL001_HEPG2_24H_X3.L2_B23:J18,POL001_HEPG2_24H_X3.L2_B23:B04,POL001_HEPG2_24H_X3.L2_B23:J17
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000121410""",4.259391,3.45476,3.75795,3.67487,4.375798,3.498071,3.253668,4.066302,4.259429,3.422022,3.174023,3.698892,3.332827,2.984651,3.111757,3.715392,3.948816,7.184917,4.098114,4.167649,5.319264,6.326293,6.697548,5.863348,6.15738,5.787116,8.092418,7.180867,7.229209,4.4642,8.170682,6.284379,7.406728,3.716609,7.517229,4.663937,…,2.672227,2.967208,2.779544,3.022542,2.918657,2.875971,2.821694,3.179074,2.703044,2.640418,2.777147,3.089819,2.49099,2.732576,2.959515,3.007277,2.565651,2.852891,2.888047,2.65674,3.184114,2.795784,2.815416,2.944061,2.772785,2.81195,2.942914,2.882027,3.285295,2.860934,3.047303,2.70038,2.835562,3.094146,2.596588,2.716797,2.886917
"""ENSG00000148584""",5.02895,5.701091,4.588117,5.62955,4.31662,4.908123,4.921504,5.323095,5.173924,4.887266,3.928919,5.655976,5.012664,5.132788,4.219748,5.365309,5.30384,6.560016,4.750107,4.659698,5.20681,5.098155,5.018672,5.778746,4.435904,5.127375,6.219457,6.31927,6.243583,3.965511,6.198976,3.887662,6.101072,4.635123,5.606835,6.208144,…,4.974272,4.987451,5.17595,4.931371,4.765587,4.958194,4.933106,4.743725,5.000041,4.467065,4.423732,5.023893,4.268767,4.845041,4.68034,5.34398,4.788734,4.665911,4.924009,4.783952,5.243553,4.715166,5.187104,5.26611,4.791726,4.16482,4.676994,5.125788,4.719841,4.58,5.071605,4.660334,4.420251,5.365908,4.593176,5